In [1]:
import os
import json
from datetime import date
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

# 1. Define structured output for SOW

In [2]:
class SOW(BaseModel):
    introduction: str = Field(description="Purpose and background of the project.")
    scope_of_work: str = Field(description="Detailed scope of the project.")
    deliverables: list[str] = Field(description="List of deliverables expected.")
    timeline: list[str] = Field(description="Major milestones and timeline.")
    roles_responsibilities: dict = Field(description="Roles and responsibilities of stakeholders.")
    budget: str = Field(description="Budget and resource allocation.")
    acceptance_criteria: list[str] = Field(description="Criteria for project acceptance.")
    start_date: date = Field(description="Project start date.")

# 2. Initialize LLM

In [3]:
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.3)

# 3. Define parser

In [4]:
parser = PydanticOutputParser(pydantic_object=SOW)

# 4. Define SOW prompt

In [5]:
sow_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a senior project manager. 
    Create a very detailed Statement of Work (SOW) for the project described below. 
    Include the following sections:
        1. Introduction
        2. Scope of Work
        3. Deliverables
        4. Timeline
        5. Roles & Responsibilities
        6. Budget
        7. Acceptance Criteria
    Requirements:
        - Each section must be approximately 1000 words.
        - Ensure all dates are in the year 2026.
        - Ensure all details are in-depth, comprehensive, and professional.
    {format_instructions}"""),
    ("human", "Project goal: {project_goal}")
]).partial(format_instructions=parser.get_format_instructions())

# 5. Build chain

In [6]:
sow_chain = sow_prompt | llm | parser

# 6. Run chain

In [7]:
#project_goal_input = "Develop a basic e-commerce website using Python and Django."
project_goal_input = input("Enter a subject: ")
project_start_date = date(2026, 2, 6)

response = sow_chain.invoke({
    "project_goal": project_goal_input,
    "start_date": str(project_start_date)
})


Enter a subject:  Skillheads Learning management system


# 7. Print structured SOW

In [8]:
print("\n--- Project Statement of Work (SOW) ---\n")
print(f"Introduction:\n{response.introduction}\n")
print(f"Scope of Work:\n{response.scope_of_work}\n")

print("Deliverables:")
for d in response.deliverables:
    print(f"- {d}")

print("\nTimeline:")
for t in response.timeline:
    print(f"- {t}")

print("\nRoles & Responsibilities:")
for role, resp in response.roles_responsibilities.items():
    print(f"- {role}: {resp}")

print(f"\nBudget:\n{response.budget}\n")

print("Acceptance Criteria:")
for c in response.acceptance_criteria:
    print(f"- {c}")

print(f"\nProject Start Date: {response.start_date}")


--- Project Statement of Work (SOW) ---

Introduction:
The Skillheads Learning Management System project aims to develop a comprehensive online platform that will facilitate the management of learning resources, courses, assessments, and student data. The system will provide a user-friendly interface for both educators and students to interact, access learning materials, track progress, and communicate effectively. This project is crucial for enhancing the overall learning experience and improving educational outcomes.

Scope of Work:
The scope of work for the Skillheads Learning Management System project includes requirements gathering, system design, development, testing, deployment, and maintenance of the online platform. It involves creating user roles, designing intuitive interfaces, integrating multimedia content, implementing assessment tools, ensuring data security, and providing scalability for future enhancements. The project will also involve training sessions for administr